In [10]:
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from torch.nn import functional as F
import pytorch_lightning as pl
from skimage import io
import glob
import os
from torchvision.models import vgg16, VGG16_Weights
from pytorch_lightning.callbacks import EarlyStopping
from torchmetrics import Accuracy
from pytorch_lightning.callbacks import ModelCheckpoint
from torchvision import transforms
from torchvision.transforms.v2 import Compose, Resize, ToTensor

class BirdDataset(Dataset):
    def __init__(self, root_data, type, mode, image_size):
       
        self.mode = mode
        self.type = type

        if self.mode == True:
            self.image_paths = glob.glob(os.path.join(root_data, "*", "*.jpg"))
        else:
            self.image_paths = glob.glob(os.path.join(root_data, "*.jpg"))  # Nur Bilder im Ordner, keine Unterordner

        self.image_size = image_size
        
       #Für TestDaten, ändert nur Bildgröße
        self.transforms = Resize(image_size)
        
        #Für Trainingsdaten, Kann bel. verändert werden.
        self.transforms2 = Compose([
            Resize(image_size),
            transforms.RandomGrayscale(0.3),
            transforms.RandomRotation(0.3),
            transforms.RandomAdjustSharpness(0.3),
            transforms.RandomHorizontalFlip(0.3),
            transforms.RandomVerticalFlip(0.3),
            transforms.RandomPerspective(0.3),
            ])
        
        
    def __len__(self):
        return len(self.image_paths)

        def __getitem__(self, idx):
            image_path = self.image_paths[idx]
            image = io.imread(image_path)
            image = ToTensor()(image)

            if self.train:
                image = self.transforms2(image)   # mit Augmentation
            else:
                image = self.transforms(image)    # nur Resize + Normalize

            if not self.has_labels:
                return image

            label = self.__extract_label(image_path)
            return image, torch.tensor(label, dtype=torch.long)

        #Bilder anzeigen
        #image = image.permute(1, 2, 0).numpy()
        #plt.imshow(image, cmap="gray", vmin=0, vmax=255)
        #plt.show()
        
        return image, torch.tensor(label, dtype=torch.long) 

    def __extract_label(self, path):
        label_name = os.path.basename(os.path.dirname(path))
        label_map = {
            "Amsel": 0,
            "Blaumeise": 1,
            "Kohlmeise": 2,
            "Rotkehlchen": 3,
            "Erlenzeisig": 4,
            "Buchfink": 5,
            "Feldsperling": 6
        }
        return label_map[label_name]

    
    

In [11]:
# Werte 
_size = (128,128)
_batch = 16
_classes = 7
_num_workers = 0

trDataset = BirdDataset("./Vogel_bild/Train", True, True, image_size=(_size))
teDataset = BirdDataset("./Vogel_bild/Test", False, True, image_size=(_size))
prDataset = BirdDataset("./Vogel_bild/Predict", False, False, image_size=(_size))

# Check if the dataset contains any images
print(f"Length of train dataset: {len(trDataset)}")
print(f"Length of test dataset: {len(teDataset)}")
print(f"Length of predict dataset: {len(prDataset)}")


trDataset, valDataset = torch.utils.data.random_split(dataset=trDataset, lengths=[0.8,0.2])
val_dataloader = DataLoader(valDataset, batch_size=_batch)
train_dataloader = DataLoader(trDataset, batch_size=_batch, shuffle=True)
test_dataloader = DataLoader(teDataset, batch_size=_batch, shuffle=False)
predict_dataloader = DataLoader(prDataset, batch_size = 1)


#Entkommentieren um Bilder anzeigen zu lassen. 
#trDataset.__getitem__(0)
#teDataset.__getitem__(5)

Length of train dataset: 2727
Length of test dataset: 704
Length of predict dataset: 13


In [12]:
class MyNet(nn.Module):
    def __init__(self):
        super(MyNet, self).__init__()

        # Block 1
        self.conv1_1 = nn.Conv2d(3, 64, kernel_size=3, padding=1)
        self.conv1_2 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(2, 2)

        # Block 2
        self.conv2_1 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.conv2_2 = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(2, 2)

        # Block 3
        self.conv3_1 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.conv3_2 = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        self.pool3 = nn.MaxPool2d(2, 2)

        # Block 4
        self.conv4_1 = nn.Conv2d(256, 512, kernel_size=3, padding=1)
        self.conv4_2 = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        self.pool4 = nn.MaxPool2d(2, 2)

        # Block 5
        self.conv5_1 = nn.Conv2d(512, 1024, kernel_size=3, padding=1)
        self.conv5_2 = nn.Conv2d(1024, 1024, kernel_size=3, padding=1)
        self.pool5 = nn.MaxPool2d(2, 2)

        # Block 6
        self.conv6_1 = nn.Conv2d(1024, 2048, kernel_size=3, padding=1)
        self.conv6_2 = nn.Conv2d(2048, 2048, kernel_size=3, padding=1)
        self.pool6 = nn.MaxPool2d(2, 2)

        # Fully Connected layers
        self.gap = nn.AdaptiveAvgPool2d((1,1))
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(2048 * 2 * 2, 256) 
        self.dr1 = nn.Dropout(0.5) 
        self.fc2 = nn.Linear(256, _classes)

    def forward(self, x):
        #Block 1
        x = F.relu(self.conv1_2(F.relu(self.conv1_1(x))))
        x = self.pool1(x)
        
        #Block 2
        x = F.relu(self.conv2_2(F.relu(self.conv2_1(x))))
        x = self.pool2(x)
        
        #Block 3
        x = F.relu(self.conv3_2(F.relu(self.conv3_1(x))))
        x = self.pool3(x)
        
        #Block 4
        x = F.relu(self.conv4_2(F.relu(self.conv4_1(x))))
        x = self.pool4(x)
        
        #Block 5
        x = F.relu(self.conv5_2(F.relu(self.conv5_1(x))))
        x = self.pool5(x)
        
        #Block 6
        x = F.relu(self.conv6_2(F.relu(self.conv6_1(x))))
        x = self.pool6(x)
        print(f"Block 1, pool_1 output shape: {x.shape}")
        
        #Fully Connected Layers
        x = self.flatten(x)
        x = F.relu(self.fc1(x))
        x = self.dr1(x) 
        x = self.fc2(x)
        return x

In [14]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import torchmetrics
import torch.nn.functional as F
from torch import nn
from torchvision.models import vgg16, VGG16_Weights
import pytorch_lightning as pl
from torchmetrics.classification import Accuracy

class MyVGGModel(pl.LightningModule):
    def __init__(self):
        super().__init__()

        # VGG16-Modell laden
        vgg = vgg16(weights=VGG16_Weights.DEFAULT)
        layers = list(vgg.features.children())
        self.backbone = nn.Sequential(*layers)

        # Layer einfrieren
        for param in self.backbone.parameters():
            param.requires_grad = False

        # Zusätzliche Faltungsschichten nach VGG16
        self.conv1 = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(512)
        self.conv2 = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(512)

        # Fully-Connected-Layer
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(2048 * 2 * 2, 256)  # 512 Kanäle * 4 * 4 bei 128x128-Bildern
        self.bn3 = nn.BatchNorm1d(256)
        self.fc2 = nn.Linear(256, 128)
        self.bn4 = nn.BatchNorm1d(128)
        self.fc3 = nn.Linear(128, _classes)

        # Verlustfunktion und Metriken
        self.loss_fn = nn.CrossEntropyLoss()
        self.train_acc = Accuracy(task="multiclass", num_classes=_classes)
        self.val_acc = Accuracy(task="multiclass", num_classes=_classes)
        self.test_acc = Accuracy(task="multiclass", num_classes=_classes)

        # Zum Speichern von Labels und Vorhersagen für die Confusion Matrix
        self.val_preds = []
        self.val_labels = []

    def forward(self, x):
        # Backbone-Feature-Extraktion (eingefroren)
        x = self.backbone(x)

        # Zusätzliche Faltungsschichten mit Batch-Normalisierung
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))

        # Fully-Connected-Layer mit Batch-Normalisierung
        x = self.flatten(x)
        x = F.relu(self.bn3(self.fc1(x)))
        x = F.relu(self.bn4(self.fc2(x)))
        x = self.fc3(x)
        return x

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits, y)
        acc = self.train_acc(logits, y)

        self.log('train_loss', loss, prog_bar=True, on_epoch=True)
        self.log('train_accuracy', acc, prog_bar=True, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits, y)
        acc = self.val_acc(logits, y)

        # Labels und Vorhersagen sammeln für Confusion Matrix
        self.val_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
        self.val_labels.extend(y.cpu().numpy())

        self.log('val_loss', loss, prog_bar=True, on_epoch=True)
        self.log('val_accuracy', acc, prog_bar=True, on_epoch=True)
        return loss

    def test_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        acc = self.test_acc(logits, y)

        self.log('test_accuracy', acc, prog_bar=True, on_epoch=True)

    def predict_step(self, batch, batch_idx):
        x = batch
        logits = self(x)
        preds = torch.argmax(logits, dim=1)
        return preds

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters())

In [35]:
class ConfusionMatrixCallback(pl.Callback):
    def __init__(self, class_names=None):
        super().__init__()
        # Standardmäßig werden die Klassen durch ihre Indizes dargestellt
        self.class_names = class_names if class_names else [str(i) for i in range(7)]  # 7 Klassen

    def on_validation_epoch_end(self, trainer, pl_module):
        # Überprüfen, ob Vorhersagen und Labels gesammelt wurden
        if not pl_module.val_preds or not pl_module.val_labels:
            print("Warnung: Keine Vorhersagen oder Labels gesammelt!")
            return

        # Confusion Matrix berechnen
        cm = confusion_matrix(pl_module.val_labels, pl_module.val_preds, labels=range(7))  # Labels für 7 Klassen
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=self.class_names)

        # Confusion Matrix visualisieren
        fig, ax = plt.subplots(figsize=(10, 8))
        disp.plot(cmap=plt.cm.Blues, ax=ax, xticks_rotation=45) 
        plt.title("Confusion Matrix")
        plt.title("WhatTheBird Confusion Matrix", fontsize=18)
        plt.xlabel("Predicted label", fontsize=16)
        plt.ylabel("Richtiges label", fontsize=16)
        plt.tight_layout()
        plt.show()

        # Speicher leeren für die nächste Epoche
        pl_module.val_preds = []
        pl_module.val_labels = []

In [ ]:
class_names = ["Amsel", "Blaumeise", "Buchfink", "Erlenzeisig", "Feldsperling","Kohlmeise","Rotkehlchen"]  # Passe die Klassennamen an

# Callbacks
es = EarlyStopping(monitor='val_loss', patience=5, verbose=True)
mc = ModelCheckpoint(monitor='val_loss', dirpath='checkpoint/', filename='bestes_model', save_top_k=1, verbose=True)
cm_callback = ConfusionMatrixCallback(class_names=class_names)

# Trainer erstellen
trainer = pl.Trainer(
    max_epochs=20,
    callbacks=[es, mc, cm_callback],  # Füge den Confusion Matrix Callback hinzu
    accelerator = "gpu"
)


# Training und Test
model = MyVGGModel()
trainer.fit(model, train_dataloader, val_dataloader)
trainer.test(model, dataloaders=test_dataloader)
predictions = trainer.predict(dataloaders=predict_dataloader)




In [ ]:
richtige_labels= [0,1,5,4,6,2,3,6,1,1,6,0,0]
i = 0

print(predictions[0])

def checkIfRight():
    if predictions[i] == richtige_labels[i]:
        return True

    else:
        return False

for batch_preds in predictions:
    print(f"Bild {i}: Ergebnis = {batch_preds.item()}, Richtig? {checkIfRight()}" )  
    i=i+1



In [22]:
print(torch.cuda.is_available())

True
